# SKU-Node Correlation Analysis

Exploratory analysis of correlations between pricing, inventory, and sales at the SKU-Node-Date level.

**Business questions:**
- How do NLC price changes affect subsequent sales volume?
- What factors most influence sales at the SKU-Node level?
- What margin levels maximize total revenue/profit?

## 1. Parameters & Setup

In [ ]:
# === PARAMETERS ===
ANALYSIS_DAYS = 30
END_DATE = "2026-03-25"
TOP_SKU_PCT = 0.90
SKU_SALES_LOOKBACK_DAYS = 60
ROLLING_WINDOW = 7
INVENTORY_SAMPLE_FREQ = "4D"
WAREHOUSE_ADDRESSES_PATH = r"C:\Users\valen\Desktop\WalmartPricing\Warehouse Addresses 03-25-2026 01-43-16 PM.csv"
ROLLBACKS_PATH = None  # Set to rollbacks Excel path to exclude rollback SKUs

In [ ]:
import sys
import os
import logging
import warnings
from datetime import timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

from src.data.loader import DataLoader
from src.adapters.module_loader import load_yaml, ensure_modules_path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
# Derived date boundaries
end_dt = pd.to_datetime(END_DATE)
start_dt = end_dt - pd.Timedelta(days=ANALYSIS_DAYS - 1)
warmup_dt = start_dt - pd.Timedelta(days=ROLLING_WINDOW)
sku_filter_start_dt = end_dt - pd.Timedelta(days=SKU_SALES_LOOKBACK_DAYS)

START_DATE = start_dt.strftime("%Y-%m-%d")
WARMUP_DATE = warmup_dt.strftime("%Y-%m-%d")
SKU_FILTER_START = sku_filter_start_dt.strftime("%Y-%m-%d")

print(f"Analysis window: {START_DATE} to {END_DATE}")
print(f"Warmup start (for rolling): {WARMUP_DATE}")
print(f"SKU filter lookback start: {SKU_FILTER_START}")

In [ ]:
# Initialize loader and configs
loader = DataLoader()
nlc_config = load_yaml("nlc_model.yaml")

## 2. SKU Filtering (top 90% by qty sold)

In [ ]:
# Load 2-month sales for SKU filtering
df_sales_filter = loader.load("dw_walmart_sales", start_date=SKU_FILTER_START)
print(f"Sales rows (last {SKU_SALES_LOOKBACK_DAYS} days): {len(df_sales_filter):,}")

In [ ]:
# Identify top SKUs by cumulative quantity sold
df_sku_qty = (
    df_sales_filter.groupby("sku")["quantity"]
    .sum()
    .reset_index()
    .rename(columns={"quantity": "total_qty"})
    .sort_values("total_qty", ascending=False)
    .reset_index(drop=True)
)
df_sku_qty["cum_qty"] = df_sku_qty["total_qty"].cumsum()
df_sku_qty["cum_pct"] = df_sku_qty["cum_qty"] / df_sku_qty["total_qty"].sum()

# Include the first SKU that crosses the threshold
cutoff_idx = (df_sku_qty["cum_pct"] <= TOP_SKU_PCT).sum()
top_skus = set(df_sku_qty.iloc[: cutoff_idx + 1]["sku"].tolist())

print(f"Total unique SKUs with sales: {df_sku_qty.shape[0]:,}")
print(f"Top {TOP_SKU_PCT:.0%} SKUs selected: {len(top_skus):,}")
print(f"Coverage: {df_sku_qty.iloc[:cutoff_idx + 1]['total_qty'].sum() / df_sku_qty['total_qty'].sum():.1%}")

In [ ]:
# Optionally exclude rollback SKUs
if ROLLBACKS_PATH:
    df_rollbacks = loader.load("rollbacks", rollbacks_path=ROLLBACKS_PATH)
    df_rollbacks["End date"] = pd.to_datetime(df_rollbacks["End date"])
    active_rollbacks = df_rollbacks[df_rollbacks["End date"] > start_dt]["Product Code"].unique()
    before = len(top_skus)
    top_skus -= set(active_rollbacks)
    print(f"Excluded {before - len(top_skus)} rollback SKUs, {len(top_skus)} remaining")
else:
    print("Rollback exclusion skipped (ROLLBACKS_PATH not set)")

## 3. Load Daily Sales

In [ ]:
# Load sales for analysis window + 7-day warmup
df_sales_raw = loader.load("dw_walmart_sales", start_date=WARMUP_DATE)
df_sales_raw["order_date"] = pd.to_datetime(df_sales_raw["order_date"])

# Filter to top SKUs and analysis+warmup window
df_sales_raw = df_sales_raw[
    (df_sales_raw["sku"].isin(top_skus))
    & (df_sales_raw["order_date"] <= end_dt)
].copy()

print(f"Sales rows after filtering: {len(df_sales_raw):,}")

In [ ]:
# Aggregate to SKU-Node-Date level
df_sales_agg = (
    df_sales_raw
    .groupby(["sku", "externalwarehouseid", "order_date"])
    .agg(
        qty_sold=("quantity", "sum"),
        revenue=("total_inv_amount", "sum"),
        profit=("profit", "sum"),
    )
    .reset_index()
    .rename(columns={"externalwarehouseid": "node", "order_date": "date"})
)
df_sales_agg["node"] = df_sales_agg["node"].astype(str)

# Valid SKU-Nodes: any with at least 1 sale in the full window
sku_nodes = df_sales_agg[["sku", "node"]].drop_duplicates()
print(f"Unique SKU-Nodes with sales: {len(sku_nodes):,}")
print(f"Daily aggregated rows: {len(df_sales_agg):,}")

## 4. Build Date Scaffold

In [ ]:
# Full date range including warmup
all_dates = pd.date_range(warmup_dt, end_dt, freq="D")
df_dates = pd.DataFrame({"date": all_dates})

# Cross-join: every SKU-Node x every date
scaffold = sku_nodes.merge(df_dates, how="cross")
print(f"Scaffold shape: {scaffold.shape[0]:,} rows ({len(sku_nodes):,} SKU-Nodes x {len(all_dates)} days)")

# Left-join sales
scaffold = scaffold.merge(df_sales_agg, on=["sku", "node", "date"], how="left")
scaffold["qty_sold"] = scaffold["qty_sold"].fillna(0)
scaffold["revenue"] = scaffold["revenue"].fillna(0)
scaffold["profit"] = scaffold["profit"].fillna(0)

print(f"Non-zero sales days: {(scaffold['qty_sold'] > 0).sum():,} / {len(scaffold):,}")

## 5. DSV History (Cost to Walmart)

In [ ]:
import re

# List all DSV files and parse dates
dsv_config = loader.get_source_config("walmart_dsv_folder")
dsv_files = loader.google.get_folder_files(dsv_config["id"])

dsv_files["parsed_date"] = dsv_files["Name"].apply(
    lambda x: re.search(r"\d{4}-\d{2}-\d{2}", str(x))
)
dsv_files = dsv_files[dsv_files["parsed_date"].notna()].copy()
dsv_files["dsv_date"] = dsv_files["parsed_date"].apply(lambda m: pd.to_datetime(m.group()))
dsv_files = dsv_files.drop(columns=["parsed_date"])

# Filter to files relevant to our window (warmup_dt - 30 days buffer to END_DATE)
buffer_start = warmup_dt - pd.Timedelta(days=30)
dsv_files = dsv_files[
    (dsv_files["dsv_date"] >= buffer_start) & (dsv_files["dsv_date"] <= end_dt)
].sort_values("dsv_date")

# Deduplicate: keep only the LATEST file per date (later upload supersedes)
dsv_files = dsv_files.sort_values("Name").groupby("dsv_date").last().reset_index()
dsv_files = dsv_files.sort_values("dsv_date")

print(f"DSV files in range (deduplicated): {len(dsv_files)}")
print(dsv_files[["Name", "dsv_date"]].to_string(index=False))


In [ ]:
# Load each DSV file, filtering to top SKUs immediately to manage memory
# Each raw file has ~2M rows; filtering to top 6K SKUs reduces to ~125K each
dsv_snapshots = []
for idx, (_, row) in enumerate(dsv_files.iterrows()):
    df_dsv_i = loader.google.get_file_as_df(
        row["ID"], "csv",
        read_cols=["SKU", "Price", "Source"],
        dtype={"SKU": str, "Price": float, "Source": str},
    )
    # Filter to top SKUs immediately to save memory
    df_dsv_i = df_dsv_i[df_dsv_i["SKU"].isin(top_skus)].copy()
    df_dsv_i = df_dsv_i.drop_duplicates(subset=["SKU", "Source"], keep="first")
    df_dsv_i["dsv_date"] = row["dsv_date"]
    dsv_snapshots.append(df_dsv_i)
    if (idx + 1) % 10 == 0 or idx == 0:
        logger.info("Loaded DSV %d/%d: %s (%d rows after SKU filter)",
                     idx + 1, len(dsv_files), row["Name"], len(df_dsv_i))

df_dsv_all = pd.concat(dsv_snapshots, ignore_index=True)
df_dsv_all = df_dsv_all.rename(columns={"SKU": "sku", "Price": "cost_to_walmart", "Source": "source"})
df_dsv_all["source"] = df_dsv_all["source"].astype(str)

print(f"Total DSV rows (filtered to top SKUs): {len(df_dsv_all):,}")
print(f"Unique DSV dates: {df_dsv_all['dsv_date'].nunique()}")


In [ ]:
# For each scaffold row, assign cost_to_walmart from the most recent DSV
# Strategy: for each analysis date, find the latest DSV date <= that date

# Step 1: Node-specific prices (source = node identifier)
df_dsv_node = df_dsv_all[df_dsv_all["source"] != "nan"].copy()
df_dsv_node = df_dsv_node.rename(columns={"source": "node"})
df_dsv_node = df_dsv_node.sort_values("dsv_date")

# Step 2: National prices (source = nan/null) as fallback
df_dsv_national = df_dsv_all[df_dsv_all["source"] == "nan"].copy()
df_dsv_national = df_dsv_national[["sku", "cost_to_walmart", "dsv_date"]].sort_values("dsv_date")

# merge_asof for node-specific prices
scaffold = scaffold.sort_values("date")
df_dsv_node = df_dsv_node.sort_values("dsv_date")

scaffold = pd.merge_asof(
    scaffold.sort_values("date"),
    df_dsv_node[["sku", "node", "cost_to_walmart", "dsv_date"]].sort_values("dsv_date"),
    left_on="date",
    right_on="dsv_date",
    by=["sku", "node"],
    direction="backward",
    suffixes=("", "_dsv"),
)

# Fill NaN cost_to_walmart with national price fallback
scaffold_missing = scaffold[scaffold["cost_to_walmart"].isna()].copy()
if len(scaffold_missing) > 0:
    scaffold_missing = scaffold_missing.drop(columns=["cost_to_walmart", "dsv_date"])
    scaffold_missing = pd.merge_asof(
        scaffold_missing.sort_values("date"),
        df_dsv_national.rename(columns={"cost_to_walmart": "cost_to_walmart_nat", "dsv_date": "dsv_date_nat"}),
        left_on="date",
        right_on="dsv_date_nat",
        by="sku",
        direction="backward",
    )
    # Update scaffold with national fallback
    scaffold.loc[scaffold["cost_to_walmart"].isna(), "cost_to_walmart"] = (
        scaffold_missing["cost_to_walmart_nat"].values
    )

scaffold = scaffold.drop(columns=["dsv_date"], errors="ignore")

pct_with_price = scaffold["cost_to_walmart"].notna().mean()
print(f"Rows with cost_to_walmart: {pct_with_price:.1%}")
print(f"Rows missing price: {scaffold['cost_to_walmart'].isna().sum():,}")

## 6. Walmart Offer Prices

In [ ]:
# Load item report (single snapshot for END_DATE)
# Note: assumes offer prices are relatively stable over the 30-day window
df_item_report = loader.load("dw_walmart_item_report", date_str=END_DATE)

# Keep relevant columns and filter to top SKUs
df_offer = df_item_report[["Product Code", "offer_price"]].copy()
df_offer = df_offer.rename(columns={"Product Code": "sku"})
df_offer = df_offer[df_offer["sku"].isin(top_skus)].drop_duplicates(subset="sku", keep="first")
df_offer["offer_price"] = pd.to_numeric(df_offer["offer_price"], errors="coerce")

print(f"Offer prices loaded: {len(df_offer):,} SKUs")

# Merge to scaffold
scaffold = scaffold.merge(df_offer, on="sku", how="left")
print(f"Rows with offer_price: {scaffold['offer_price'].notna().sum():,} / {len(scaffold):,}")

## 7. Supporting Data

In [ ]:
# MAP prices
df_map = loader.load("dw_map_prices", date_str=END_DATE)
df_map = df_map.rename(columns={"Product Code": "sku"})
df_map = df_map[["sku", "MAP"]].drop_duplicates(subset="sku", keep="first")
df_map["MAP"] = pd.to_numeric(df_map["MAP"], errors="coerce")
scaffold = scaffold.merge(df_map, on="sku", how="left")
print(f"SKUs with MAP: {scaffold['MAP'].notna().sum():,} rows")

# Shipping costs
df_shipping = loader.load("shipping_costs_by_node")
df_shipping = df_shipping.rename(columns={"Identifier": "node"})
df_shipping["node"] = df_shipping["node"].astype(str)
scaffold = scaffold.merge(df_shipping[["node", "Shipping cost"]], on="node", how="left")
scaffold = scaffold.rename(columns={"Shipping cost": "shipping_cost"})
print(f"Rows with shipping cost: {scaffold['shipping_cost'].notna().sum():,}")

In [ ]:
# Warehouse node mapping (for bridging Warehouse Code <-> Identifier)
df_wh_mapping = loader.load("warehouse_node_mapping")
df_wh_mapping["Identifier"] = df_wh_mapping["Identifier"].astype(str)
df_wh_mapping["Warehouse Code"] = df_wh_mapping["Warehouse Code"].astype(str)
node_to_wh = df_wh_mapping[["Identifier", "Warehouse Code"]].drop_duplicates(subset="Identifier")

# Warehouse addresses -> city/state
df_addresses = pd.read_csv(WAREHOUSE_ADDRESSES_PATH, dtype={"Code": str})
df_city = df_addresses[["Code", "Town", "State"]].rename(columns={"Code": "Warehouse Code"})

# Join: node -> Warehouse Code -> Town/State
node_city = node_to_wh.merge(df_city, on="Warehouse Code", how="left")
node_city = node_city.rename(columns={"Identifier": "node"})

scaffold = scaffold.merge(node_city[["node", "Town", "State"]], on="node", how="left")
print(f"Rows with city mapping: {scaffold['Town'].notna().sum():,} / {len(scaffold):,}")

## 8. Inventory (sampled every 4 days)

In [ ]:
# Lazy import pricing_module
ensure_modules_path()
import pricing_module as pricing

# Sample dates across the window
inv_dates = pd.date_range(start_dt, end_dt, freq=INVENTORY_SAMPLE_FREQ).strftime("%Y-%m-%d").tolist()
# Ensure END_DATE is included
if END_DATE not in inv_dates:
    inv_dates.append(END_DATE)

print(f"Inventory dates to load: {len(inv_dates)}")
print(inv_dates)

In [ ]:
# Load inventory for sampled dates
min_units = nlc_config["inventory"]["min_units_secondary"]  # 4

# Warehouse node mapping for filtering (WalmartB2B, ENABLED, Inventory Enabled=1)
df_wh_filt = df_wh_mapping[
    (df_wh_mapping["Channel"] == "WalmartB2B")
    & (df_wh_mapping["Warehouse Status"] == "ENABLED")
    & (df_wh_mapping["Identifier Status"] == "ENABLED")
    & (df_wh_mapping["Inventory Enabled"] == 1)
].copy()

inv_records = []
for date_i in inv_dates:
    logger.info("Loading inventory for %s...", date_i)
    df_inv_i = pricing.get_inventory(date_i, add_rebates=False, amazon=False, greater3=True)
    df_inv_i["date"] = pd.to_datetime(date_i)
    
    # Filter by min_units
    df_inv_i = df_inv_i[df_inv_i["Available"] >= min_units].copy()
    
    # Filter to top SKUs
    df_inv_i = df_inv_i[df_inv_i["Product Code"].isin(top_skus)].copy()
    
    # Merge with node mapping
    df_inv_i["Warehouse Code"] = df_inv_i["Warehouse Code"].astype(str)
    df_inv_i = df_inv_i.merge(
        df_wh_filt[["Warehouse Code", "Identifier", "Inventory Threshold"]],
        on="Warehouse Code", how="inner"
    )
    
    # Filter by Inventory Threshold
    df_inv_i = df_inv_i[df_inv_i["Available"] >= df_inv_i["Inventory Threshold"]].copy()
    
    # Per SKU-Node: min Purchase Price+FET
    df_inv_agg = (
        df_inv_i.groupby(["Product Code", "Identifier"])
        .agg({"Purchase Price+FET": "min"})
        .reset_index()
        .rename(columns={
            "Product Code": "sku",
            "Identifier": "node",
            "Purchase Price+FET": "min_purchase_price_fet",
        })
    )
    df_inv_agg["inv_date"] = pd.to_datetime(date_i)
    df_inv_agg["can_show_inv"] = 1
    inv_records.append(df_inv_agg)
    logger.info("  -> %d SKU-Node pairs with inventory", len(df_inv_agg))

df_inv_all = pd.concat(inv_records, ignore_index=True)
df_inv_all["node"] = df_inv_all["node"].astype(str)
print(f"Total inventory records: {len(df_inv_all):,}")


In [ ]:
# Forward-fill inventory to all analysis dates using merge_asof
scaffold = scaffold.sort_values("date")
df_inv_all = df_inv_all.sort_values("inv_date")

scaffold = pd.merge_asof(
    scaffold,
    df_inv_all[["sku", "node", "min_purchase_price_fet", "can_show_inv", "inv_date"]],
    left_on="date",
    right_on="inv_date",
    by=["sku", "node"],
    direction="backward",
)
scaffold["can_show_inv"] = scaffold["can_show_inv"].fillna(0).astype(int)
scaffold = scaffold.drop(columns=["inv_date"], errors="ignore")

print(f"Rows with inventory: {(scaffold['can_show_inv'] == 1).sum():,} / {len(scaffold):,}")
print(f"Rows with purchase price: {scaffold['min_purchase_price_fet'].notna().sum():,}")

## 9. Assemble Master DataFrame

In [ ]:
# Computed columns
scaffold["walmart_margin"] = (
    (scaffold["offer_price"] - scaffold["cost_to_walmart"]) / scaffold["offer_price"]
)
scaffold["te_margin"] = (
    (scaffold["cost_to_walmart"] - scaffold["min_purchase_price_fet"]) / scaffold["cost_to_walmart"]
)

# Brand = first 4 chars of SKU
scaffold["brand"] = scaffold["sku"].str[:4]

# Day of week
scaffold["day_of_week"] = scaffold["date"].dt.dayofweek  # 0=Mon, 6=Sun

# MAP proximity: how close cost_to_walmart is to the MAP floor
scaffold["map_proximity"] = scaffold["cost_to_walmart"] / (scaffold["MAP"] * 0.95)

# Number of active nodes per SKU per date
active_nodes = (
    scaffold[scaffold["can_show_inv"] == 1]
    .groupby(["sku", "date"])["node"]
    .nunique()
    .reset_index()
    .rename(columns={"node": "n_active_nodes"})
)
scaffold = scaffold.merge(active_nodes, on=["sku", "date"], how="left")
scaffold["n_active_nodes"] = scaffold["n_active_nodes"].fillna(0).astype(int)

print("Computed columns added.")
print(f"Columns: {scaffold.columns.tolist()}")

In [ ]:
# Days since last price change (from DSV history)
# Build a per-SKU-Node price change timeline from DSV snapshots
dsv_prices = (
    df_dsv_all[df_dsv_all["source"] != "nan"]
    .rename(columns={"source": "node_dsv"})
    .sort_values("dsv_date")
)

# Detect price changes per SKU-Node
dsv_prices["prev_price"] = dsv_prices.groupby(["sku", "node_dsv"])["cost_to_walmart"].shift(1)
dsv_prices["price_changed"] = (
    (dsv_prices["cost_to_walmart"] != dsv_prices["prev_price"])
    & dsv_prices["prev_price"].notna()
)

# Get change dates
change_dates = dsv_prices[dsv_prices["price_changed"]][["sku", "node_dsv", "dsv_date"]].copy()
change_dates = change_dates.rename(columns={"node_dsv": "node", "dsv_date": "change_date"})
change_dates = change_dates.sort_values("change_date")

# For each scaffold row, find most recent price change
if len(change_dates) > 0:
    scaffold = scaffold.sort_values("date")
    scaffold = pd.merge_asof(
        scaffold,
        change_dates,
        left_on="date",
        right_on="change_date",
        by=["sku", "node"],
        direction="backward",
    )
    scaffold["days_since_price_change"] = (scaffold["date"] - scaffold["change_date"]).dt.days
    scaffold = scaffold.drop(columns=["change_date"], errors="ignore")
else:
    scaffold["days_since_price_change"] = np.nan

print(f"Rows with days_since_price_change: {scaffold['days_since_price_change'].notna().sum():,}")

## 10. Rolling 7-Day Comparisons

In [ ]:
# Sort by SKU-Node-Date for rolling calculations
scaffold = scaffold.sort_values(["sku", "node", "date"]).reset_index(drop=True)

rolling_cols = ["qty_sold", "te_margin", "cost_to_walmart", "offer_price", "walmart_margin"]

for col in rolling_cols:
    # Rolling 7-day average of prior days (shift 1 to exclude current day)
    scaffold[f"{col}_7d_avg"] = (
        scaffold.groupby(["sku", "node"])[col]
        .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW, min_periods=1).mean())
    )
    # Delta and percentage change
    scaffold[f"{col}_vs_7d"] = scaffold[col] - scaffold[f"{col}_7d_avg"]
    scaffold[f"{col}_vs_7d_pct"] = (
        scaffold[f"{col}_vs_7d"] / scaffold[f"{col}_7d_avg"].replace(0, np.nan)
    )

print(f"Rolling comparison columns added for: {rolling_cols}")

In [ ]:
# Trim warmup period â€” keep only the analysis window
df = scaffold[scaffold["date"] >= start_dt].copy().reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nColumn types:")
print(df.dtypes)

## 11. Exploratory Data Analysis

In [ ]:
# Dataset summary
print(f"Shape: {df.shape}")
print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print(f"\nDescriptive statistics:")
df.describe()

In [ ]:
# Missing values heatmap
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(df.isnull().T, cbar=False, cmap="YlOrRd", ax=ax)
ax.set_title("Missing Values by Column")
plt.tight_layout()
plt.show()

In [ ]:
# Spearman correlation matrix (full numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove ID-like columns
numeric_cols = [c for c in numeric_cols if c not in ["day_of_week"]]

corr_matrix = df[numeric_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, ax=ax,
    annot_kws={"size": 7}, square=True,
)
ax.set_title("Spearman Rank Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with qty_sold
target_corrs = (
    corr_matrix["qty_sold"]
    .drop("qty_sold")
    .dropna()
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in target_corrs.head(15)]
target_corrs.head(15).plot.barh(ax=ax, color=colors)
ax.set_xlabel("Spearman Correlation with qty_sold")
ax.set_title("Top 15 Features Correlated with Sales Volume")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nTop 15 correlations with qty_sold:")
print(target_corrs.head(15).to_string())

In [ ]:
# Scatter plots for top 6 correlated features (individual figures for clarity)
top_features = target_corrs.head(6).index.tolist()

# Sample for performance
df_nonzero = df[df["qty_sold"] > 0]
df_sample = df_nonzero.sample(n=min(10000, len(df_nonzero)), random_state=42)

for col in top_features:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.regplot(
        data=df_sample, x=col, y="qty_sold", ax=ax,
        scatter_kws={"alpha": 0.2, "s": 8}, line_kws={"color": "red"},
    )
    valid = df_sample[[col, "qty_sold"]].dropna()
    r, p = stats.spearmanr(valid[col], valid["qty_sold"])
    ax.set_title(f"{col} vs qty_sold  (Spearman r={r:.3f}, p={p:.2e})", fontsize=13)
    ax.set_xlabel(col, fontsize=11)
    ax.set_ylabel("qty_sold", fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
# Distribution plots for key variables (individual figures)
dist_cols = ["qty_sold", "walmart_margin", "te_margin", "cost_to_walmart", "days_since_price_change"]
dist_cols = [c for c in dist_cols if c in df.columns]

for col in dist_cols:
    fig, ax = plt.subplots(figsize=(10, 5))
    data = df[col].dropna()
    if col == "qty_sold":
        data = data[data > 0]  # Skip zeros for visibility
        ax.set_title(f"Distribution of {col} (non-zero only)", fontsize=13)
    else:
        ax.set_title(f"Distribution of {col}", fontsize=13)
    sns.histplot(data, ax=ax, bins=50, kde=True)
    ax.set_xlabel(col, fontsize=11)
    plt.tight_layout()
    plt.show()


## 12. Statistical Tests

In [ ]:
# Test 1: Price change impact on sales
# Compare qty_sold on days with a recent price change (<=3 days) vs no change (>7 days)
recent_change = df[df["days_since_price_change"] <= 3]["qty_sold"]
no_change = df[df["days_since_price_change"] > 7]["qty_sold"]

if len(recent_change) > 0 and len(no_change) > 0:
    stat, pval = stats.mannwhitneyu(recent_change, no_change, alternative="two-sided")
    print("=== Price Change Impact on Sales ===")
    print(f"Recent change (<=3 days): n={len(recent_change):,}, mean={recent_change.mean():.3f}, median={recent_change.median():.1f}")
    print(f"No recent change (>7 days): n={len(no_change):,}, mean={no_change.mean():.3f}, median={no_change.median():.1f}")
    print(f"Mann-Whitney U stat={stat:.0f}, p-value={pval:.4e}")
    print(f"Significant at 5%: {'Yes' if pval < 0.05 else 'No'}")
else:
    print("Insufficient data for price change analysis")

In [ ]:
# Test 2: Margin level vs. revenue
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, label in zip(axes, ["te_margin", "walmart_margin"], ["TE Margin", "Walmart Margin"]):
    df_valid = df[df[col].notna() & (df["qty_sold"] > 0)].copy()
    if len(df_valid) > 100:
        df_valid["margin_bin"] = pd.qcut(df_valid[col], 10, duplicates="drop")
        margin_revenue = (
            df_valid.groupby("margin_bin", observed=True)
            .agg(avg_qty=("qty_sold", "mean"), total_qty=("qty_sold", "sum"), count=("qty_sold", "count"))
            .reset_index()
        )
        margin_revenue["margin_bin"] = margin_revenue["margin_bin"].astype(str)
        ax.bar(range(len(margin_revenue)), margin_revenue["avg_qty"])
        ax.set_xticks(range(len(margin_revenue)))
        ax.set_xticklabels(margin_revenue["margin_bin"], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Avg Qty Sold")
        ax.set_title(f"Avg Sales by {label} Decile")
    else:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# Test 3: Inventory availability vs sales
inv_yes = df[df["can_show_inv"] == 1]["qty_sold"]
inv_no = df[df["can_show_inv"] == 0]["qty_sold"]

print("=== Inventory Availability vs Sales ===")
print(f"With inventory:    n={len(inv_yes):,}, mean={inv_yes.mean():.3f}, median={inv_yes.median():.1f}")
print(f"Without inventory: n={len(inv_no):,}, mean={inv_no.mean():.3f}, median={inv_no.median():.1f}")

if len(inv_yes) > 0 and len(inv_no) > 0:
    stat, pval = stats.mannwhitneyu(inv_yes, inv_no, alternative="two-sided")
    print(f"Mann-Whitney U stat={stat:.0f}, p-value={pval:.4e}")
    print(f"Significant at 5%: {'Yes' if pval < 0.05 else 'No'}")

In [ ]:
# Test 4: OLS regression
import statsmodels.api as sm

feature_cols = [
    "cost_to_walmart", "offer_price", "walmart_margin", "te_margin",
    "can_show_inv", "shipping_cost", "n_active_nodes", "day_of_week",
    "days_since_price_change",
]
feature_cols = [c for c in feature_cols if c in df.columns]

df_reg = df[feature_cols + ["qty_sold"]].dropna()

if len(df_reg) > 100:
    X = sm.add_constant(df_reg[feature_cols])
    y = df_reg["qty_sold"]
    model = sm.OLS(y, X).fit()
    print(model.summary())
else:
    print("Insufficient data for OLS regression")

In [ ]:
# Also run with log-transformed qty_sold (to handle zero-inflation)
df_reg_log = df_reg[df_reg["qty_sold"] > 0].copy()
if len(df_reg_log) > 100:
    X_log = sm.add_constant(df_reg_log[feature_cols])
    y_log = np.log1p(df_reg_log["qty_sold"])
    model_log = sm.OLS(y_log, X_log).fit()
    print("=== OLS with log(1 + qty_sold), non-zero sales only ===")
    print(model_log.summary())
else:
    print("Insufficient non-zero sales data for log OLS")

### Test 5: Price Change Impact on Revenue

Study the correlation between Walmart price changes and revenue directly,
complementing the qty-based analysis in Test 1.

In [ ]:
# Price Change Impact on Revenue
if "revenue" in df.columns and "cost_to_walmart_pct_chg" in df.columns:
    df_pc = df[df["cost_to_walmart_pct_chg"].notna() & df["revenue"].notna()].copy()
    df_pc = df_pc[df_pc["cost_to_walmart_pct_chg"] != 0]

    if len(df_pc) > 50:
        from scipy.stats import spearmanr, pearsonr, mannwhitneyu

        r_pearson, p_pearson = pearsonr(df_pc["cost_to_walmart_pct_chg"], df_pc["revenue"])
        r_spearman, p_spearman = spearmanr(df_pc["cost_to_walmart_pct_chg"], df_pc["revenue"])
        print("=== Price Change vs Revenue Correlation ===")
        print(f"Pearson  r = {r_pearson:.4f}  (p = {p_pearson:.4g})")
        print(f"Spearman r = {r_spearman:.4f}  (p = {p_spearman:.4g})")

        # Bucketed analysis
        df_pc["price_change_bucket"] = pd.cut(
            df_pc["cost_to_walmart_pct_chg"],
            bins=[-np.inf, -0.05, -0.01, 0.01, 0.05, np.inf],
            labels=["Large decrease (<-5%)", "Small decrease (-5% to -1%)",
                    "Minimal (-1% to 1%)", "Small increase (1% to 5%)",
                    "Large increase (>5%)"],
        )
        bucket_stats = (
            df_pc.groupby("price_change_bucket", observed=True)
            .agg(n=("revenue", "size"), mean_revenue=("revenue", "mean"),
                 median_revenue=("revenue", "median"), total_revenue=("revenue", "sum"),
                 mean_qty=("qty_sold", "mean"))
            .round(2)
        )
        print()
        print("=== Revenue by Price Change Bucket ===")
        print(bucket_stats.to_string())

        # Visualization
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].scatter(df_pc["cost_to_walmart_pct_chg"] * 100, df_pc["revenue"], alpha=0.15, s=8)
        axes[0].set_xlabel("Price Change (%)")
        axes[0].set_ylabel("Revenue ($)")
        axes[0].set_title(f"Price Change vs Revenue (r={r_spearman:.3f})")
        axes[0].axvline(0, color="red", linestyle="--", alpha=0.5)

        bucket_stats["mean_revenue"].plot(kind="bar", ax=axes[1], color="steelblue", alpha=0.8)
        axes[1].set_ylabel("Mean Revenue ($)")
        axes[1].set_title("Mean Revenue by Price Change Bucket")
        axes[1].tick_params(axis="x", rotation=30)

        bucket_stats["mean_qty"].plot(kind="bar", ax=axes[2], color="darkorange", alpha=0.8)
        axes[2].set_ylabel("Mean Qty Sold")
        axes[2].set_title("Mean Qty by Price Change Bucket")
        axes[2].tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.show()

        # Pre/post revenue around price change events
        print()
        print("=== Pre/Post Revenue Around Price Change Events ===")
        df_events = df[df["days_since_price_change"].notna()].copy()
        pre_rev = df_events[df_events["days_since_price_change"].between(4, 7)]["revenue"]
        post_rev = df_events[df_events["days_since_price_change"].between(0, 3)]["revenue"]
        print(f"Pre-change (4-7 days before):  n={len(pre_rev):,}, mean=${pre_rev.mean():.2f}")
        print(f"Post-change (0-3 days after):  n={len(post_rev):,}, mean=${post_rev.mean():.2f}")
        if len(pre_rev) > 30 and len(post_rev) > 30:
            stat, pval = mannwhitneyu(post_rev, pre_rev, alternative="two-sided")
            print(f"Mann-Whitney U test: stat={stat:.1f}, p={pval:.4g}")
            pct_change = (post_rev.mean() - pre_rev.mean()) / pre_rev.mean() * 100
            print(f"Revenue change: {pct_change:+.1f}%")
    else:
        print(f"Insufficient price-change rows ({len(df_pc)}) for revenue correlation analysis")
else:
    missing = [c for c in ["revenue", "cost_to_walmart_pct_chg"] if c not in df.columns]
    print(f"Columns missing for price-revenue analysis: {missing}")


## 13. Geographic & Brand Analysis

In [ ]:
# Sales by state
state_sales = (
    df.groupby("State")
    .agg(
        total_qty=("qty_sold", "sum"),
        avg_wm_margin=("walmart_margin", "mean"),
        avg_te_margin=("te_margin", "mean"),
        n_nodes=("node", "nunique"),
    )
    .sort_values("total_qty", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(14, 6))
state_sales["total_qty"].plot.bar(ax=ax)
ax.set_title("Total Sales by State (Top 20)")
ax.set_ylabel("Total Qty Sold")
ax.set_xlabel("State")
plt.tight_layout()
plt.show()

state_sales

In [ ]:
# Brand-level analysis
brand_analysis = (
    df.groupby("brand")
    .agg(
        total_qty=("qty_sold", "sum"),
        total_revenue=("revenue", "sum"),
        avg_te_margin=("te_margin", "mean"),
        avg_wm_margin=("walmart_margin", "mean"),
        n_skus=("sku", "nunique"),
        n_nodes=("node", "nunique"),
    )
    .sort_values("total_qty", ascending=False)
    .head(20)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

brand_analysis["total_qty"].plot.bar(ax=axes[0])
axes[0].set_title("Total Sales by Brand (Top 20)")
axes[0].set_ylabel("Total Qty Sold")

brand_analysis[["avg_te_margin", "avg_wm_margin"]].plot.bar(ax=axes[1])
axes[1].set_title("Average Margins by Brand (Top 20)")
axes[1].set_ylabel("Margin")
axes[1].legend(["TE Margin", "Walmart Margin"])

plt.tight_layout()
plt.show()

brand_analysis

In [ ]:
# Node distribution breadth vs sales
breadth = (
    df.groupby("sku")
    .agg(
        avg_active_nodes=("n_active_nodes", "mean"),
        total_qty=("qty_sold", "sum"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=breadth, x="avg_active_nodes", y="total_qty", alpha=0.4, s=15, ax=ax)
ax.set_xlabel("Avg Active Nodes per SKU")
ax.set_ylabel("Total Qty Sold (30 days)")
ax.set_title("Distribution Breadth vs. Sales Volume")

r, p = stats.spearmanr(breadth["avg_active_nodes"], breadth["total_qty"])
ax.text(0.05, 0.95, f"Spearman r={r:.3f}, p={p:.2e}", transform=ax.transAxes, va="top")

plt.tight_layout()
plt.show()

## 14. Summary & Cleanup

### Key Findings

*(Fill in after running the analysis)*

1. **Price change impact:** ...
2. **Top correlated features:** ...
3. **Margin sweet spots:** ...
4. **Inventory availability effect:** ...
5. **Geographic patterns:** ...
6. **Brand insights:** ...
7. **Distribution breadth:** ...

In [ ]:
# Save the assembled dataset for future use
project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
output_dir = os.path.join(project_root, "outputs")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"correlation_dataset_{END_DATE}.parquet")
try:
    df.to_parquet(output_path, index=False)
    print(f"Dataset saved to {output_path}")
except Exception as e:
    output_path = output_path.replace(".parquet", ".csv")
    df.to_csv(output_path, index=False)
    print(f"Parquet failed ({e}), saved as CSV: {output_path}")

print(f"Shape: {df.shape}")

# Cleanup
loader.close()
print("Done.")
